# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The notebook uses the package's default OpenAI `gpt-4.1-mini` client. Set `OPENAI_API_KEY` in the environment before running the model cell. The key is read by the OpenAI SDK and is never stored in this notebook.

In [1]:
from pathlib import Path

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=2, cache_dir=ROOT / 'data' / 'raw')
[(story.splitlines()[0], len(story)) for story in stories]

[('THE DAW IN BORROWED FEATHERS', 859), ('THE SUN AND THE WIND', 972)]

In [2]:
len(stories), [story.splitlines()[0] for story in stories]

(2, ['THE DAW IN BORROWED FEATHERS', 'THE SUN AND THE WIND'])

## OpenAI model

The transformer creates the default `OpenAIModelClient` when `model` is omitted. It uses `gpt-4.1-mini` through the Responses API and reads `OPENAI_API_KEY` from the environment.

In [3]:
graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
)
traces = graphicalizer.fit([stories[0]]).transform_with_trace([stories[0]])
trace = traces[0]
[(p.text, p.proposition_id) for p in trace.propositions]

[SemanticGraphicalizer] ready: model=OpenAIModelClient, ontology=aesop-narrative, domain=aesop-narrative
[document-0] processing document: chars=859
[document-0] segment: 1 -> 1 | 0.1 ms | input_chars=859, chunk_chars=859
[document-0 chunk-0] compiling semantic stages
[document-0 chunk-0] summarize: 1 -> 1 | 3142.8 ms | input_chars=859, output_chars=480
[document-0 chunk-0] normalize: 1 -> 1 | 1881.2 ms | input_chars=480, output_chars=769
[document-0 chunk-0] decompose: 1 -> 13 | 9723.8 ms
[document-0 chunk-0] triple: 13 -> 13 | 12369.1 ms
[document-0] integrate: 13 -> 13 | 0.3 ms | nodes=16, edges=13
[document-0] total: 1 -> 1 | 27117.5 ms | chunks=1, propositions=13, triples=13, nodes=16, edges=13


[('A jackdaw has the trait of vanity.', 'document-0:chunk-0:p1'),
 ('The jackdaw performs the action of adorning himself with colorful peacock feathers.',
  'document-0:chunk-0:p2'),
 ('The jackdaw does this to achieve the goal of appearing as beautiful as peacocks.',
  'document-0:chunk-0:p3'),
 ('The action of adorning with colorful peacock feathers occurs in a place where peacocks are found.',
  'document-0:chunk-0:p4'),
 ('The jackdaw interacts with the peacocks by attempting to mingle with them.',
  'document-0:chunk-0:p5'),
 ("The peacocks detect the jackdaw's unnatural behavior.",
  'document-0:chunk-0:p6'),
 ('The peacocks perform the action of attacking and removing the borrowed feathers from the jackdaw.',
  'document-0:chunk-0:p7'),
 ('The jackdaw is driven away by the peacocks after they remove the borrowed feathers.',
  'document-0:chunk-0:p8'),
 ('The jackdaw returns to his original group.', 'document-0:chunk-0:p9'),
 ('The jackdaw interacts with his old companions.', 'do

In [4]:
graphicalizer.display(trace.graph)

<IPython.core.display.Javascript object>